# 🛡️ Sentinel AI Incident Commander — Cloud GPU Inference Server

Run the fine-tuned **Sentinel-Coder-7B (Q4_K_M GGUF)** model on a free **16 GB NVIDIA T4 GPU** in Google Colab.

- **Hugging Face Model Repository**: [`kamaleshkumarR/sentinel-ggu`](https://huggingface.co/kamaleshkumarR/sentinel-ggu)
- **Model File**: `sentinel-q4_k_m.gguf`
- **Hardware Acceleration**: 100% CUDA GPU Layer Offload (T4 16GB VRAM)

### ⚡ Step 1: Install `llama-cpp-python` with CUDA GPU Acceleration

In [ ]:
# Install llama-cpp with CUDA GPU acceleration and cloudflared
!CMAKE_ARGS="-DGGML_CUDA=on" pip install -U "llama-cpp-python[server]"
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("✅ CUDA GPU Environment and Cloudflared installed successfully!")

### 🧠 Step 2: Load Sentinel Model into the 16 GB GPU

In [ ]:
from llama_cpp import Llama

print("[*] Downloading and loading Sentinel Q4_K_M model from kamaleshkumarR/sentinel-ggu...")
llm = Llama.from_pretrained(
    repo_id="kamaleshkumarR/sentinel-ggu",
    filename="sentinel-q4_k_m.gguf",
    n_gpu_layers=-1,   # Offload 100% of layers to NVIDIA T4 GPU
    n_ctx=4096,        # Context window for stack traces & AST call graphs
    verbose=True
)
print("\n✅ Sentinel model successfully loaded into GPU VRAM!")

### 🧪 Step 3: Run Live Incident Triage Diagnosis (Interactive Test)

In [ ]:
SYSTEM_PROMPT = """You are Sentinel, an autonomous AI Incident Commander and Senior SRE. 
Analyze the incident telemetry, trace execution flow, identify root cause, and output structured JSON."""

USER_INCIDENT = """SERVICE: checkout-api
ERROR: OperationalError: connection pool exhausted timeout (pool_size=20, max_overflow=10)
STACK TRACE:
  File "app/routes/checkout.py", line 84, in create_order
  File "app/services/payment.py", line 112, in process_transaction
  File "app/database.py", line 45, in get_connection
CLUSTER COUNT: 3 repeated occurrences in last 5 minutes."""

print("[*] Generating diagnosis with fine-tuned Sentinel model on GPU...")
response = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_INCIDENT}
    ],
    temperature=0.1,
    max_tokens=512
)

print("\n" + "="*60)
print("🛡️ SENTINEL AI DIAGNOSIS OUTPUT:")
print("="*60)
print(response["choices"][0]["message"]["content"])
print("="*60)

### 🌐 Step 4: Start 24/7 OpenAI-Compatible API Server & Expose via Cloudflare Tunnel

Run this cell to start an OpenAI-compatible API server (`/v1/chat/completions`) and get a public HTTPS link to plug into **Render** or your live frontend dashboard.

In [ ]:
import subprocess
import time

print("[*] Launching OpenAI-compatible API server for kamaleshkumarR/sentinel-ggu on GPU...")
server = subprocess.Popen([
    "python3", "-m", "llama_cpp.server",
    "--hf_model_repo_id", "kamaleshkumarR/sentinel-ggu",
    "--model_alias", "sentinel",
    "--n_gpu_layers", "-1",
    "--n_ctx", "4096",
    "--host", "0.0.0.0",
    "--port", "8000"
])

# Wait 12 seconds for server to initialize in background
time.sleep(12)

print("\n[*] Opening secure Cloudflare HTTPS Tunnel...")
!./cloudflared tunnel --url http://localhost:8000